<a href="https://colab.research.google.com/github/nikkhilbhargav/laptop-price-prediction/blob/main/laptop_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


url = "https://raw.githubusercontent.com/nikkhilbhargav/laptop-price-prediction/refs/heads/main/data.csv"

In [ ]:


df = pd.read_csv(url)

df.drop(columns=["Unnamed: 0.1", "Unnamed: 0", "name"], inplace=True)

df.info()
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 893 entries, 0 to 892
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand              893 non-null    object 
 1   price              893 non-null    int64  
 2   spec_rating        893 non-null    float64
 3   processor          893 non-null    object 
 4   CPU                893 non-null    object 
 5   Ram                893 non-null    object 
 6   Ram_type           893 non-null    object 
 7   ROM                893 non-null    object 
 8   ROM_type           893 non-null    object 
 9   GPU                893 non-null    object 
 10  display_size       893 non-null    float64
 11  resolution_width   893 non-null    float64
 12  resolution_height  893 non-null    float64
 13  OS                 893 non-null    object 
 14  warranty           893 non-null    int64  
dtypes: float64(4), int64(2), object(9)
memory usage: 104.8+ KB
    brand  pric

In [ ]:
pd.DataFrame({'count': df.shape[0],
              'nulls': df.isnull().sum(),
              'nulls%': df.isnull().mean() * 100,
              'cardinality': df.nunique(),
             })

,count,nulls,nulls%,cardinality
brand,893,0,0.0,30
price,893,0,0.0,464
spec_rating,893,0,0.0,30
processor,893,0,0.0,184
CPU,893,0,0.0,29
Ram,893,0,0.0,7
Ram_type,893,0,0.0,12
ROM,893,0,0.0,7
ROM_type,893,0,0.0,2
GPU,893,0,0.0,134


In [ ]:
# GET NUM OF COLOMS AND ROWS
print(f"Number of column :{df.shape[1]}\nNumber of rows :{df.shape[0]}")

Number of column :15
Number of rows :893


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np

In [ ]:
x = df.drop(["price"],axis=1)
y = df['price']


In [ ]:
categorical_features = x.select_dtypes(include=["object"]).columns
print(categorical_features)

Index(['brand', 'processor', 'CPU', 'Ram', 'Ram_type', 'ROM', 'ROM_type',
       'GPU', 'OS'],
      dtype='object')


In [ ]:
x_train , x_test , y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

print("x_ train :\n", x_train)
print("y_ train :\n", y_train)

x_ train :
        brand  spec_rating                      processor  \
331       HP    67.000000    5th Gen AMD Ryzen 5  5600H    
734       HP    69.323529  Intel Pentium Silver   N6000    
382     Dell    69.323529      5th Gen AMD Ryzen 5 5500U   
705  Infinix    62.000000  11th Gen Intel Core i7 1195G7   
814      MSI    75.000000     7th Gen AMD Ryzen 5 7535HS   
..       ...          ...                            ...   
106     Asus    69.323529   12th Gen Intel Core i3 1220P   
270    Wings    69.323529  11th Gen Intel Core i7 1165G7   
860     Dell    69.323529   13th Gen Intel Core i3 1305U   
435     Asus    69.323529  11th Gen Intel Core i3 1115G4   
102     Asus    60.000000  11th Gen Intel Core i5 1135G7   

                                CPU   Ram Ram_type    ROM ROM_type  \
331           Hexa Core, 12 Threads   8GB     DDR4  512GB      SSD   
734            Quad Core, 4 Threads   8GB     DDR4  512GB      SSD   
382           Hexa Core, 12 Threads   8GB     DDR4  512GB

In [ ]:
cat_features = [
    x.columns.get_loc(col)
    for col in categorical_features
]
print(cat_features)

[0, 2, 3, 4, 5, 6, 7, 8, 12]


In [ ]:
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

In [ ]:
model.fit(
    x_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 59883.4418758	total: 56.9ms	remaining: 28.4s
100:	learn: 21032.1232925	total: 1.67s	remaining: 6.58s
200:	learn: 15044.6934173	total: 2.67s	remaining: 3.96s
300:	learn: 11448.3962662	total: 3.69s	remaining: 2.44s
400:	learn: 9644.6969453	total: 4.13s	remaining: 1.02s
499:	learn: 8421.3439959	total: 4.56s	remaining: 0us


CatBoostRegressor(depth=6, iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
print("R² Score :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("MSE :", mean_squared_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))

R² Score : 0.858789450935668
MAE : 12683.95263617961
MSE : 483878363.609763
RMSE : 21997.235362876014


In [ ]:
import pandas as pd

new_laptop = pd.DataFrame({
    "brand": ["Macbook"],
    "spec_rating": [75],
    "processor": ["Core i5"],
    "CPU": ["12th Gen"],
    "Ram": ["16 GB"],
    "Ram_type": ["DDR4"],
    "ROM": ["512 GB"],
    "ROM_type": ["SSD"],
    "GPU": ["os"],
    "display_size": [15.6],
    "resolution_width": [1920],
    "resolution_height": [1080],
    "OS": ["Windows 11"],
    "warranty": [1]
})


predicted_price = model.predict(new_laptop)

print(f"Predicted Laptop Price: ₹{predicted_price[0]:,.2f}")

Predicted Laptop Price: ₹55,386.96


In [ ]:
import joblib

joblib.dump(model, "catboost_model.pkl")

['catboost_model.pkl']